# HCC1806 Karyotyping Results Analysis

- Use environment.yml

# Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib import rcParams
import pathlib
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import FuncFormatter, MultipleLocator

# Functions

In [ ]:
def plot_clone_violins(
    rep_df,
    sample_order=None,
    week_order=(0, 5),
    week_colors=None,
    show_violin=True,
    jitter=0.03,
    figsize=(14, 6),
    title=None,
    ylabel="Count",
    use_ploidy_axis=False,
    ploidy_unit=23,   # 1N = 23
    max_ploidy=3      # show up to 3N by default
):
    """
    rep_df: wide DataFrame, columns like 'C1_week0', 'C1_week5', ...
            rows = replicates
    sample_order: list of sample names in desired order
                  e.g. ['C1','C2',...,'C9','F1',...,'F9','GFP','MC']
    week_order: tuple/list of week values in order (default (0,5))
    week_colors: dict mapping week -> color, e.g. {0:'blue', 5:'red'}
    show_violin: if True, show violin silhouette + quartile line;
                 if False, only show quartile line + points.
    jitter: horizontal jitter size for individual data points.
    use_ploidy_axis: if True, y-axis ticks shown as 1N, 2N, 3N...
    ploidy_unit: numeric value corresponding to 1N (default 23).
    max_ploidy: highest N to show (e.g. 5 -> 1N..5N).
    """
    for font_path in pathlib.Path('/stor/work/Brock/kennedy/fonts/arial').glob('*.TTF'):
        font_manager.fontManager.addfont(str(font_path))

    arial_n = {'fontproperties':font_manager.FontProperties(fname="../misc/arial/ARIALN.TTF")}
    arial_nb = {'fontproperties':font_manager.FontProperties(fname="../misc/arial/ARIALNB.TTF")}

    assert 'Arial' in [f.name for f in font_manager.fontManager.ttflist]

    rcParams['font.family'] = 'Arial'

    # --- reshape rep_df to long format ---
    long = (
        rep_df.reset_index(drop=True)
              .reset_index(names="rep")  # rep column 0,1,2...
              .melt(id_vars="rep", var_name="sample_week", value_name="value")
    )
    long = long.dropna(subset=["value"])

    # split 'C1_week0' -> sample='C1', week=0
    sample_week_split = long["sample_week"].str.rsplit("_week", n=1, expand=True)
    long["sample"] = sample_week_split[0]
    long["week"] = sample_week_split[1].astype(int)

    # default orders
    if sample_order is None:
        sample_order = sorted(long["sample"].unique())

    if week_colors is None:
        week_colors = {week_order[0]: "tab:blue", week_order[1]: "tab:red"}

    # x positions: one base position per sample,
    # week0 at x - offset, week5 at x + offset
    n_samples = len(sample_order)
    base_x = np.arange(n_samples)
    offset = 0.18

    pos_map = {}
    for i, s in enumerate(sample_order):
        for w_i, w in enumerate(week_order):
            pos = base_x[i] + (-offset if w_i == 0 else offset)
            pos_map[(s, w)] = pos

    fig, ax = plt.subplots(figsize=figsize)

    # --- draw violins / quartile lines / points ---
    rng = np.random.default_rng(42)  # reproducible jitter

    for (sample, week), group in long.groupby(["sample", "week"]):
        if sample not in sample_order or week not in week_order:
            continue

        x = pos_map[(sample, week)]
        vals = group["value"].to_numpy()
        color = week_colors.get(week, "gray")

        # Violin silhouette (optional)
        if show_violin:
            v = ax.violinplot(
                vals,
                positions=[x],
                widths=0.28,
                showextrema=False,
                showmeans=False,
                showmedians=False,
            )
            for body in v["bodies"]:
                body.set_facecolor(color)
                body.set_edgecolor(color)
                body.set_alpha(0.5)

        # Quartiles
        q1, med, q3 = np.percentile(vals, [25, 50, 75])
        ax.vlines(x, q1, q3, color=color, linewidth=2)
        ax.scatter([x, x, x], [q1, med, q3], color=color, s=20, zorder=4)

        # Individual data points (black)
        x_jitter = x + rng.uniform(-jitter, jitter, size=len(vals))
        ax.scatter(x_jitter, vals, color="black", s=10, alpha=0.7, zorder=3)

    # --- axis cosmetics ---
    ax.set_xticks(base_x)
    ax.set_xticklabels(sample_order)
    ax.set_xlim(base_x[0] - 0.6, base_x[-1] + 0.6)

    # Y-axis: numeric or N-style labels
    if use_ploidy_axis:
        # Hard cap at the chosen max_ploidy (e.g. 5N)
        ylim_top = ploidy_unit * max_ploidy
        ax.set_ylim(0, ylim_top)

        ticks = [ploidy_unit * n for n in range(1, max_ploidy + 1)]
        labels = [f"{n}N" for n in range(1, max_ploidy + 1)]
        ax.set_yticks(ticks)
        ax.set_yticklabels(labels)
        ax.set_ylabel("Ploidy")
    else:
        ax.set_ylabel(ylabel)

    # Legend
    handles = []
    labels = []
    for w in week_order:
        if w in week_colors:
            h = ax.scatter([], [], color=week_colors[w], label=f"Week {w}")
            handles.append(h)
            labels.append(f"Week {w}")
    ax.legend(handles, labels, title="Timepoint", frameon=False)

    plt.tight_layout()

    if title:
        plt.title(title)

    return fig, ax

In [ ]:
def stats_and_plot(dataframe, alpha=0.05, xlabel=None, ylabel=None, title=None, 
                    sig_bars=True, point_label=False, fig_size=(10, 6), font_sz=18, 
                    save_svg=False, ploidy_axis=False, chromosomes_per_n=23,
                    correction='bonferroni'):
    """
    Perform pairwise statistical tests between each column of the DataFrame and 
    create a box and whisker plot with significance indicators.

    For each pair of columns, automatically detects whether the data are paired 
    (same row index/subject present in both columns, e.g. same clones measured at 
    two timepoints) or independent, and selects the appropriate test:
        Paired:      Paired t-test (if differences are normal) or Wilcoxon signed-rank
        Independent: t-test / Welch's t-test / Mann-Whitney U, based on normality 
                     and variance checks

    All pairwise raw p-values are then corrected together using either FDR 
    (Benjamini-Hochberg) or Bonferroni.

    Parameters:
    dataframe (pd.DataFrame): Input data, one column per group/condition. Row index 
        should identify the subject/clone so paired comparisons can be detected.
    alpha (float): Significance threshold. Default 0.05.
    correction (str): 'fdr' (Benjamini-Hochberg, default) or 'bonferroni'.
    ploidy_axis (bool): If True, format y-axis ticks as "1N", "2N", etc.
    chromosomes_per_n (float): Value representing 1N in the data's units 
        (e.g. 23 for raw chromosome counts, 1 if data is already in N units).
    save_svg (bool): If True, saves the plot as an SVG using the title as filename.

    Returns:
    pd.DataFrame: Symmetric matrix of corrected (adjusted) p-values between each 
        pair of columns.
    """
    import matplotlib as mpl
    from itertools import combinations
    from scipy.stats import (shapiro, levene, mannwhitneyu, ttest_ind, 
                              ttest_rel, wilcoxon)
    from statsmodels.stats.multitest import multipletests
    from matplotlib.ticker import FuncFormatter, MultipleLocator

    assert correction in ('fdr', 'bonferroni'), "correction must be 'fdr' or 'bonferroni'"

    dataframe = dataframe.apply(pd.to_numeric, errors='coerce')
    columns = list(dataframe.columns)
    num_samples = len(columns)

    fig, ax = plt.subplots(figsize=fig_size)

    # --- Clean spine styling ---
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

    # --- Boxplot ---
    bp = ax.boxplot(
        [dataframe[col].dropna() for col in columns],
        labels=columns,
        showmeans=True,
        showfliers=False,
        patch_artist=True,
        meanprops=dict(marker='D', markerfacecolor='black', markeredgecolor='black', markersize=5),
        medianprops=dict(color='black', linewidth=2),
        boxprops=dict(facecolor='white', color='black', linewidth=1.5),
        whiskerprops=dict(color='black', linewidth=1.5, linestyle='--'),
        capprops=dict(color='black', linewidth=1.5),
    )

    # --- Jittered data points ---
    for i, col in enumerate(columns, start=1):
        y = dataframe[col].dropna()
        x = np.random.normal(i, 0.04, len(y))
        ax.scatter(x, y, color='dimgray', alpha=0.6, s=30, zorder=3, edgecolors='none')

        if point_label:
            valid_indices = dataframe[col].dropna().index
            for x_val, y_val, label in zip(x, y, valid_indices):
                ax.text(x_val, y_val, str(label), fontsize=8, ha='right', va='bottom')

    # --- Overall assumption testing (reported for reference; per-pair checks done below) ---
    alpha_assumption = 0.05
    groups = [dataframe[col].dropna().values for col in columns]
    normality_ok = all(shapiro(g).pvalue > alpha_assumption for g in groups if len(g) >= 3)
    levene_p = levene(*groups).pvalue if len(groups) > 1 else 1.0
    variance_ok = levene_p > alpha_assumption

    print(f"Overall normality (Shapiro-Wilk):     {'PASS' if normality_ok else 'FAIL'}")
    print(f"Overall equal variance (Levene's):    {'PASS' if variance_ok else 'FAIL'}")
    print(f"Correction method:                    {'Benjamini-Hochberg (FDR)' if correction == 'fdr' else 'Bonferroni'}")
    print()

    # --- Pairwise comparisons: adaptively choose paired vs. independent test per pair ---
    pair_results = []

    for col1, col2 in combinations(columns, 2):
        s1, s2 = dataframe[col1], dataframe[col2]
        idx1, idx2 = s1.dropna().index, s2.dropna().index
        shared_idx = idx1.intersection(idx2)

        is_paired = (len(shared_idx) >= 2) and (set(idx1) == set(idx2))

        if is_paired:
            v1 = s1.loc[shared_idx].values
            v2 = s2.loc[shared_idx].values
            diffs = v1 - v2

            diff_normal = shapiro(diffs).pvalue > alpha_assumption if len(diffs) >= 3 else False

            if diff_normal:
                test_used = "Paired t-test"
                _, p_raw = ttest_rel(v1, v2)
            else:
                test_used = "Wilcoxon signed-rank"
                try:
                    _, p_raw = wilcoxon(v1, v2)
                except ValueError:
                    test_used = "Wilcoxon signed-rank (degenerate, no variance)"
                    p_raw = 1.0
        else:
            g1, g2 = s1.dropna().values, s2.dropna().values
            if len(g1) < 2 or len(g2) < 2:
                continue  # not enough data to compare

            n1_ok = shapiro(g1).pvalue > alpha_assumption if len(g1) >= 3 else False
            n2_ok = shapiro(g2).pvalue > alpha_assumption if len(g2) >= 3 else False
            pair_normal = n1_ok and n2_ok
            pair_var_ok = levene(g1, g2).pvalue > alpha_assumption

            if pair_normal and pair_var_ok:
                test_used = "Independent t-test"
                _, p_raw = ttest_ind(g1, g2, equal_var=True)
            elif pair_normal and not pair_var_ok:
                test_used = "Welch's t-test"
                _, p_raw = ttest_ind(g1, g2, equal_var=False)
            else:
                test_used = "Mann-Whitney U"
                _, p_raw = mannwhitneyu(g1, g2, alternative='two-sided')

        pair_results.append({'col1': col1, 'col2': col2, 'p_raw': p_raw,
                              'test': test_used, 'paired': is_paired})

    # --- Apply multiple comparison correction across ALL pairs together ---
    p_values_df = pd.DataFrame(np.nan, columns=columns, index=columns)

    if pair_results:
        raw_pvals = [r['p_raw'] for r in pair_results]
        method = 'fdr_bh' if correction == 'fdr' else 'bonferroni'
        _, p_adj, _, _ = multipletests(raw_pvals, alpha=alpha, method=method)

        for r, p_corrected in zip(pair_results, p_adj):
            r['p_adj'] = p_corrected
            p_values_df.loc[r['col1'], r['col2']] = p_corrected
            p_values_df.loc[r['col2'], r['col1']] = p_corrected

            tag = "PAIRED" if r['paired'] else "unpaired"
            sig = " *" if p_corrected < alpha else ""
            print(f"{r['col1']} vs {r['col2']:<20s} [{tag:8s}] {r['test']:<28s} "
                  f"raw p={r['p_raw']:.4f}  adj p={p_corrected:.4f}{sig}")

    # --- Significance bars ---
    if sig_bars and pair_results:
        y_max = dataframe.max().max()
        y_min_ax, y_max_ax = ax.get_ylim()
        y_span = y_max_ax - y_min_ax
        line_offset = y_span * 0.03
        text_offset = y_span * 0.01
        highest_y = y_max

        for r in pair_results:
            p_adj = r['p_adj']
            if p_adj < alpha:
                if p_adj < 0.001:
                    sig_symbol = '***'
                elif p_adj < 0.01:
                    sig_symbol = '**'
                else:
                    sig_symbol = '*'

                x1 = columns.index(r['col1']) + 1
                x2 = columns.index(r['col2']) + 1
                y = y_max + line_offset

                ax.plot([x1, x2], [y, y], lw=1.5, color='black')
                ax.text((x1 + x2) * 0.5, y + text_offset, sig_symbol,
                        ha='center', va='bottom', color='black', fontsize=13)

                line_offset += y_span * 0.09
                highest_y = y + text_offset

        ax.set_ylim(bottom=dataframe.min().min() * 0.95, top=highest_y * 1.02)

    # --- Ploidy-style y-axis labels (integer N only, e.g. "2N", "3N", "4N") ---
    if ploidy_axis:
        def ploidy_formatter(x, pos):
            n_value = round(x / chromosomes_per_n)
            return f"{n_value}N"
        ax.yaxis.set_major_locator(MultipleLocator(chromosomes_per_n))
        ax.yaxis.set_major_formatter(FuncFormatter(ploidy_formatter))

    # --- Labels & title ---
    ax.set_xlabel(xlabel if xlabel else 'Samples', labelpad=10)
    ax.set_ylabel(ylabel if ylabel else 'Growth Rate', labelpad=10)
    plot_title = title if title else 'Boxplot with Statistical Significance Indicators'
    ax.set_title(plot_title, pad=12, fontweight='bold')

    fig.tight_layout()

    if save_svg:
        filename = f"{plot_title}.svg"
        fig.savefig(filename, format='svg', bbox_inches='tight', dpi=300)

    plt.show()
    return p_values_df

In [ ]:
def build_ploidy_summary(df):
    """
    Collapse a wide dataframe of raw chromosome counts (columns like 'C1_week0', 
    'F3_week5', 'GFP_week5', etc.) into a summary dataframe of per-clone means,
    reshaped into group/timepoint columns for use with stats_and_plot().
    """
    # Mean of each raw column, ignoring NaNs (missing replicates)
    means = df.mean(skipna=True)

    records = {}
    for col, val in means.items():
        sample, timepoint = col.rsplit('_', 1)  # e.g. 'C1', 'week0'

        if sample.startswith('C'):
            group = 'Control'
            tp_label = 'Initial' if timepoint == 'week0' else 'Week 5'
            new_col = f"{group} {tp_label}"
        elif sample.startswith('F'):
            group = 'Fusion'
            tp_label = 'Initial' if timepoint == 'week0' else 'Week 5'
            new_col = f"{group} {tp_label}"
        elif sample in ('GFP', 'MC'):
            new_col = f"{sample}_{timepoint}"  # e.g. 'GFP_week5', 'MC_week5'
        else:
            raise ValueError(f"Unrecognized sample prefix in column: {col}")

        records.setdefault(sample, {})[new_col] = val

    summary_df = pd.DataFrame.from_dict(records, orient='index')

    # Enforce a sensible column order (only keep columns that actually exist)
    col_order = ['Control Initial', 'Control Week 5', 
                 'Fusion Initial', 'Fusion Week 5', 
                 'GFP_week5', 'MC_week5']
    col_order = [c for c in col_order if c in summary_df.columns]
    summary_df = summary_df[col_order]

    # Sort rows naturally: C1-C9, then F1-F9, then GFP, MC
    summary_df = summary_df.reindex(sorted(summary_df.index, 
                                            key=lambda s: (s[0] not in 'CF', s)))

    return summary_df

In [ ]:
def make_clone_summary(rep_df, agg="median"):
    """
    Convert wide rep_df to a clone-level summary:
    index = sample (C1, F3, etc.)
    columns = week (0, 5, ...)
    values = median (or mean) over cells.
    """
    long = (
        rep_df.reset_index(drop=True)
              .reset_index(names="rep")
              .melt(id_vars="rep", var_name="sample_week", value_name="value")
    )
    long = long.dropna(subset=["value"])

    # split 'C1_week0' → sample='C1', week=0
    sw = long["sample_week"].str.rsplit("_week", n=1, expand=True)
    long["sample"] = sw[0]
    long["week"]   = sw[1].astype(int)

    if agg == "median":
        summary = long.groupby(["sample", "week"])["value"].median().unstack("week")
    elif agg == "mean":
        summary = long.groupby(["sample", "week"])["value"].mean().unstack("week")
    else:
        raise ValueError("agg must be 'median' or 'mean'")

    return summary  # rows: samples, cols: weeks

# Main

In [ ]:
# --- 1. Load CSVs ---

# main counts file (image_name, count, clone, week, etc.)
counts = pd.read_csv("/stor/work/Brock/kennedy/SC_repo/data/Ploidy/HCC1806_ploidy/ChromosomeCounts_5weekupdate.csv")

# metadata file (name, id, clone) where "id" is F1, F2, ... and "clone" is 1806_01, etc.
meta   = pd.read_csv("/stor/work/Brock/kennedy/SC_repo/data/Ploidy/HCC1806_ploidy/HCC1806_new-clone-ID.csv")

# keep only needed columns
counts = counts[['image_name', 'count', 'clone', 'week']]

# remove rows with missing week, then cast to int
counts = counts[counts['week'].notna()].copy()
counts['week'] = counts['week'].astype(int)   # now only 0, 5, etc.

# --- 2. Attach the F1/F2/... IDs to each row via the clone code ---

# meta has columns: name, id, clone
df = counts.merge(meta[['clone', 'id']], on='clone', how='left')

# df now has: image_name, count, clone, week, id


# --- 3. Build the final column labels like "F5_week0", "F5_week5", etc. ---

df['sample'] = df['id'].astype(str) + "_week" + df['week'].astype(str)
# columns look like: F1_week0, F1_week5, F5_week0, F5_week5, ...


# give each (F#, week) combination a replicate number
df['rep'] = df.groupby('sample').cumcount() + 1

chrom_counts = df.pivot(
    index='rep',          # replicate number per sample
    columns='sample',
    values='count'
).sort_index()

In [ ]:
font_sz = 18
legend_font_sz = font_sz-2

rcParams['font.size'] = font_sz  # Default font size for all text

In [ ]:
sample_order = [f"C{i}" for i in range(1,9)] + \
               [f"F{i}" for i in range(1,9)] #+ ["GFP", "MC"]

fig, ax = plot_clone_violins(
    chrom_counts.iloc[:,:-2],
    sample_order=sample_order,
    week_order=(0, 5),
    week_colors={0: "dimgrey", 5: "black"},
    show_violin=False,
    use_ploidy_axis=True,  # turn on 1N, 2N, 3N... ticks
    ploidy_unit=23,        # 1N = 23
    max_ploidy=6,           # show 1N–5N
    # title='HCC1806 Fusion vs. Control Ploidy Via Karyotyping',
    title=None,
    figsize=(10, 6)
)
plt.savefig('1806ploidyViolinnotitle.svg', format='svg')
plt.show()

In [ ]:
# Make sure all values are numeric where possible
chrom_counts_num = chrom_counts.apply(pd.to_numeric, errors="coerce")

results = {
    "C_end_5_avg": chrom_counts_num.loc[:, chrom_counts_num.columns.str.startswith("C") & chrom_counts_num.columns.str.endswith("5")].stack().mean(),
    "C_end_0_avg": chrom_counts_num.loc[:, chrom_counts_num.columns.str.startswith("C") & chrom_counts_num.columns.str.endswith("0")].stack().mean(),
    "F_end_0_avg": chrom_counts_num.loc[:, chrom_counts_num.columns.str.startswith("F") & chrom_counts_num.columns.str.endswith("0")].stack().mean(),
    "F_end_5_avg": chrom_counts_num.loc[:, chrom_counts_num.columns.str.startswith("F") & chrom_counts_num.columns.str.endswith("5")].stack().mean(),
}

results_df = pd.DataFrame.from_dict(results, orient="index", columns=["Average"])
results_df

In [ ]:
exclude = ['rep','GFP_week5','MC_week5']
cols = [c for c in chrom_counts_num.columns if c not in exclude ]
avg_non_missing = chrom_counts_num[cols].count().mean()
print(f"Average non-missing values per column: {avg_non_missing:.2f}")

max_non_missing = chrom_counts_num[cols].count().max()
print(f"Max non-missing values per column: {max_non_missing:.2f}")

min_non_missing = chrom_counts_num[cols].count().min()
print(f"Min non-missing values per column: {min_non_missing:.2f}")

In [ ]:
ploidy_summary = build_ploidy_summary(chrom_counts_num)
print(ploidy_summary)

print("\nPloidy Averages")
for column in ploidy_summary.columns:
    average = ploidy_summary[column].mean()
    standard_error = ploidy_summary[column].sem()
    standard_dev = ploidy_summary[column].std()
    print(f"\n{column}")
    print("Average:", average)
    print("Standard Error:", standard_error)
    print("Standard Deviation:", standard_dev,'\n')

In [ ]:
p_values = stats_and_plot(ploidy_summary.iloc[:, :-2], alpha=0.05, sig_bars=True, point_label=False, ylabel='HCC1806 Ploidy\n(N = 23 chromosomes)', ploidy_axis=True, fig_size=(8, 6), title='HCC1806 Average ploidy Week 0 vs 5', save_svg=True)
p_values